# Model Experiment — `SARIMA`

## Theory

**SARIMA(p, d, q)(P, D, Q, s)** extends ARIMA with a seasonal component: a
second, seasonal AR/I/MA structure applied at lag `s` (here `s=52` weeks). It can
also take exogenous regressors (then usually called **SARIMAX**). This directly
targets the strong yearly seasonality (Thanksgiving/Christmas) the EDA found,
which plain ARIMA cannot represent.

**Why here:** same per-(Store, Dept) series setup as the ARIMA notebook — one
independent model per series (~3,000 series) — but now with a seasonal term and
the option to pass `IsHoliday` as an exogenous regressor, so the model can react
to a holiday week even when it doesn't line up with a full 52-week cycle yet
(e.g., the first 1-2 years of a series).

**Cost of the seasonal term:** a seasonal period of 52 needs a few years of
history to estimate reliably, and fitting is noticeably slower than plain ARIMA.
With ~143 weeks of training history per series that's only ~2.75 seasonal
cycles — workable but not generous, so series that are too short fall back to
the same recent/Dept/global-mean strategy as the ARIMA notebook.

**Per the assignment brief:** same guidance as ARIMA — theoretical discussion
matters more than exhaustive tuning here, so the search below stays small and
guided rather than a full grid.

**Tracking:** MLflow (classical statistical model, not a neural net).

## 0. Config & environment (Kaggle vs local)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_COMPETITION = 'walmart-recruiting-store-sales-forecasting'
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    RAW_DIR = KAGGLE_INPUT / KAGGLE_COMPETITION
    WORKING_DIR = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path.cwd().parent
    RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
    WORKING_DIR = PROJECT_ROOT
    load_dotenv(PROJECT_ROOT / '.env')

def _resolve(stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        p = RAW_DIR / name
        if p.exists():
            return p
    return RAW_DIR / f'{stem}.csv'

TRAIN_CSV = _resolve('train')
TEST_CSV = _resolve('test')
FEATURES_CSV = _resolve('features')
STORES_CSV = _resolve('stores')

RANDOM_SEED = 42
TARGET = 'Weekly_Sales'
HOLIDAY_WEIGHT = 5
NON_HOLIDAY_WEIGHT = 1

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for key in ('MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD'):
            try:
                os.environ.setdefault(key, client.get_secret(key))
            except Exception:
                pass
    except Exception:
        pass

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

print('On Kaggle:', ON_KAGGLE, '| raw data dir:', RAW_DIR)

## 1. Data loading helpers

In [ ]:
import pandas as pd

def _read_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})

def load_stores():
    return pd.read_csv(STORES_CSV)

def load_features():
    df = pd.read_csv(FEATURES_CSV, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    df = df.sort_values(['Store', 'Date'])
    for col in ('CPI', 'Unemployment'):
        df[col] = df.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    return df.reset_index(drop=True)

def load_raw(split):
    path = TRAIN_CSV if split == 'train' else TEST_CSV
    df = pd.read_csv(path, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    return df

def load_merged(split='train'):
    base = load_raw(split)
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    df = base.merge(stores, on='Store', how='left')
    df = df.merge(feats, on=['Store', 'Date'], how='left')
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return df

## 2. Metric (WMAE) & cross-validation helpers

In [ ]:
def weights_from_holiday(is_holiday):
    is_holiday = np.asarray(is_holiday).astype(bool)
    return np.where(is_holiday, HOLIDAY_WEIGHT, NON_HOLIDAY_WEIGHT).astype(float)

def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = weights_from_holiday(is_holiday)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

In [ ]:
def _sorted_unique_dates(dates):
    return np.sort(pd.to_datetime(dates).unique())

def time_holdout(df, weeks=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    if weeks >= len(uniq):
        raise ValueError(f'weeks={weeks} >= number of distinct weeks {len(uniq)}')
    cutoff = uniq[-weeks]
    d = pd.to_datetime(df[date_col]).to_numpy()
    train_idx = np.where(d < cutoff)[0]
    val_idx = np.where(d >= cutoff)[0]
    return train_idx, val_idx

def expanding_splits(df, n_splits=3, horizon=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    needed = horizon * n_splits
    if needed >= len(uniq):
        raise ValueError(f'Need > {needed} distinct weeks for {n_splits} folds of {horizon}; have {len(uniq)}.')
    d = pd.to_datetime(df[date_col]).to_numpy()
    for k in range(n_splits):
        end_offset = needed - k * horizon
        start_offset = end_offset - horizon
        val_start = uniq[-end_offset]
        val_end = uniq[-start_offset] if start_offset > 0 else None
        train_idx = np.where(d < val_start)[0]
        if val_end is None:
            val_idx = np.where(d >= val_start)[0]
        else:
            val_idx = np.where((d >= val_start) & (d < val_end))[0]
        yield train_idx, val_idx

## 3. Per-series ARIMA/SARIMA Pipeline building blocks

In [ ]:
import warnings
from statsmodels.tsa.arima.model import ARIMA
from sklearn.base import BaseEstimator, RegressorMixin

FREQ = 'W-FRI'

def build_series_panel(df, value_col='Weekly_Sales'):
    """{(Store,Dept): continuous weekly pd.Series}, small gaps linearly interpolated."""
    panel = {}
    for key, g in df.groupby(['Store', 'Dept']):
        g = g.sort_values('Date').drop_duplicates('Date')
        s = g.set_index('Date')[value_col]
        full_idx = pd.date_range(s.index.min(), s.index.max(), freq=FREQ)
        s = s.reindex(full_idx)
        s = s.interpolate(limit=4).ffill().bfill()
        panel[key] = s
    return panel

def build_exog_panel(df, exog_cols):
    """{(Store,Dept): continuous weekly pd.DataFrame of exog cols}, aligned to same index."""
    if not exog_cols:
        return {}
    panel = {}
    for key, g in df.groupby(['Store', 'Dept']):
        g = g.sort_values('Date').drop_duplicates('Date')
        e = g.set_index('Date')[exog_cols].astype(float)
        full_idx = pd.date_range(e.index.min(), e.index.max(), freq=FREQ)
        e = e.reindex(full_idx).ffill().bfill()
        panel[key] = e
    return panel


class PerSeriesForecaster(BaseEstimator, RegressorMixin):
    """Fits one statsmodels ARIMA/SARIMAX per (Store,Dept) series and predicts by
    aligning requested Dates to forecast steps ahead of that series' training window.
    Falls back to a recent-mean / Dept-mean / global-mean predictor for series that
    are new (cold-start in test), too short, or fail to converge."""

    def __init__(self, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), exog_cols=None, min_obs=20):
        self.order = order
        self.seasonal_order = seasonal_order
        self.exog_cols = exog_cols
        self.min_obs = min_obs

    def fit(self, X, y=None):
        df = X.copy()
        df['Weekly_Sales'] = np.asarray(y, dtype=float)
        panel = build_series_panel(df, 'Weekly_Sales')
        exog_panel = build_exog_panel(df, self.exog_cols) if self.exog_cols else {}

        self.models_ = {}
        self.last_date_ = {}
        self.fallback_value_ = {}
        n_failed = 0
        for key, s in panel.items():
            self.fallback_value_[key] = float(s.tail(8).mean()) if len(s) else 0.0
            if len(s) < self.min_obs:
                self.models_[key] = None
                self.last_date_[key] = s.index.max()
                continue
            exog = exog_panel.get(key)
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    model = ARIMA(
                        s.values, order=self.order, seasonal_order=self.seasonal_order,
                        exog=(exog.values if exog is not None else None),
                        enforce_stationarity=False, enforce_invertibility=False,
                    )
                    res = model.fit()
                self.models_[key] = res
            except Exception:
                self.models_[key] = None
                n_failed += 1
            self.last_date_[key] = s.index.max()

        self.dept_mean_ = df.groupby('Dept')['Weekly_Sales'].mean()
        self.global_mean_ = float(df['Weekly_Sales'].mean())
        self.n_series_ = len(panel)
        self.n_failed_ = n_failed
        return self

    def _fallback(self, key, store, dept):
        if key in self.fallback_value_:
            return self.fallback_value_[key]
        if dept in self.dept_mean_.index:
            return float(self.dept_mean_.loc[dept])
        return self.global_mean_

    def predict(self, X):
        preds = np.empty(len(X), dtype=float)
        X = X.reset_index(drop=True)
        for (store, dept), idx in X.groupby(['Store', 'Dept']).groups.items():
            key = (store, dept)
            rows = X.loc[idx]
            if key not in self.models_ or self.models_[key] is None:
                preds[idx] = self._fallback(key, store, dept)
                continue
            res = self.models_[key]
            last_date = self.last_date_[key]
            dates = pd.to_datetime(rows['Date'])
            steps = ((dates - last_date).dt.days // 7).to_numpy()
            max_step = int(steps.max()) if len(steps) else 0
            if max_step < 1:
                preds[idx] = self._fallback(key, store, dept)
                continue
            future_exog = None
            if self.exog_cols:
                future_exog = np.zeros((max_step, len(self.exog_cols)))
                for s, dt in zip(steps, dates):
                    if 1 <= s <= max_step:
                        match = rows.loc[rows['Date'] == dt, self.exog_cols].astype(float)
                        if len(match):
                            future_exog[s - 1] = match.values[0]
            try:
                fc_mean = res.get_forecast(steps=max_step, exog=future_exog).predicted_mean
            except Exception:
                fc_mean = np.full(max_step, self._fallback(key, store, dept))
            for pos, s in zip(idx, steps):
                if s < 1:
                    preds[pos] = self._fallback(key, store, dept)
                else:
                    preds[pos] = fc_mean[min(int(s), max_step) - 1]
        return preds

## 4. MLflow tracking setup

In [ ]:
import mlflow

def init_tracking(experiment=None):
    uri = MLFLOW_TRACKING_URI
    if uri:
        mlflow.set_tracking_uri(uri)
        if MLFLOW_TRACKING_USERNAME:
            os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME
        if MLFLOW_TRACKING_PASSWORD:
            os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD
    else:
        mlflow.set_tracking_uri((WORKING_DIR / 'mlruns').as_uri())
    if experiment:
        mlflow.set_experiment(experiment)
    return mlflow.get_tracking_uri()

import numpy as np, pandas as pd
MODEL_NAME = 'SARIMA'
EXPERIMENT = f'{MODEL_NAME}_Training'
uri = init_tracking(EXPERIMENT)
print('MLflow ->', uri, '| experiment:', EXPERIMENT)

## Load raw data (merged with IsHoliday already present in raw train/test)

In [ ]:
raw_train = load_raw('train')
raw_test  = load_raw('test')
raw_train.shape, raw_test.shape

## Run 1 — Cleaning
Same per-series diagnostics as ARIMA. A seasonal model additionally needs
enough history per series to estimate the `s=52` term, so we also report how
many series are shorter than 2 seasonal cycles (104 weeks).

In [ ]:
n_series = raw_train.groupby(['Store', 'Dept']).ngroups
lengths = raw_train.groupby(['Store', 'Dept']).size()
span_weeks = raw_train.groupby(['Store', 'Dept'])['Date'].agg(lambda d: (d.max() - d.min()).days // 7 + 1)
n_with_gaps = int((lengths < span_weeks).sum())
n_short_for_seasonal = int((lengths < 104).sum())

with mlflow.start_run(run_name=f'{MODEL_NAME}_Cleaning'):
    mlflow.log_param('n_series', n_series)
    mlflow.log_param('n_series_with_gaps', n_with_gaps)
    mlflow.log_param('n_series_shorter_than_2_seasons', n_short_for_seasonal)
    mlflow.log_param('gap_fill_strategy', 'reindex_to_weekly+interpolate(limit=4)+ffill/bfill')
    mlflow.log_param('cold_start_fallback', 'recent_mean -> dept_mean -> global_mean')
    mlflow.log_metric('n_train_rows', len(raw_train))
    mlflow.log_metric('n_negative_sales', int((raw_train.Weekly_Sales < 0).sum()))
    print(f'{n_series} series, {n_with_gaps} with gaps, {n_short_for_seasonal} shorter than 104 weeks')

## Run 2 — "Feature Selection" (seasonal order & exogenous regressor choice)
Same idea as ARIMA's Run 2, plus deciding the seasonal order `(P, D, Q, 52)` and
whether `IsHoliday` as an exogenous regressor actually helps (tested in the CV
grid below, not assumed).

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

agg = raw_train.groupby('Date')['Weekly_Sales'].sum()
seasonal_diff = agg.diff(52).dropna()
adf_level = adfuller(agg)[1]
adf_seasonal_diff = adfuller(seasonal_diff)[1]
print(f'ADF p-value, level: {adf_level:.4f} | after seasonal (52) difference: {adf_seasonal_diff:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(agg.diff().dropna(), lags=60, ax=axes[0], title='ACF (1st diff, aggregate)')
plot_pacf(agg.diff().dropna(), lags=60, ax=axes[1], title='PACF (1st diff, aggregate)')
plt.tight_layout(); plt.show()

candidate_configs = [
    {'order': (1, 1, 1), 'seasonal_order': (0, 1, 0, 52), 'exog_cols': None},
    {'order': (1, 1, 1), 'seasonal_order': (0, 1, 1, 52), 'exog_cols': None},
    {'order': (1, 1, 1), 'seasonal_order': (1, 1, 0, 52), 'exog_cols': None},
    {'order': (1, 1, 1), 'seasonal_order': (0, 1, 1, 52), 'exog_cols': ['IsHoliday']},
]

with mlflow.start_run(run_name=f'{MODEL_NAME}_Feature_Selection'):
    mlflow.log_param('adf_pvalue_level', adf_level)
    mlflow.log_param('adf_pvalue_seasonal_diff', adf_seasonal_diff)
    mlflow.log_param('seasonal_period', 52)
    mlflow.log_param('candidate_configs', candidate_configs)
print('candidate configs:', candidate_configs)

## Run 3 — Cross-validation (small guided grid)
Same runtime-conscious approach as ARIMA: a random subsample of series, one
holdout split, and note that seasonal fits are slower so the sample is a bit
smaller.

In [ ]:
SAMPLE_SIZE = 40
rng = np.random.RandomState(RANDOM_SEED)
all_keys = list(raw_train.groupby(['Store', 'Dept']).groups.keys())
long_keys = [k for k in all_keys if lengths.get(k, 0) >= 104]
pool = long_keys if len(long_keys) >= SAMPLE_SIZE else all_keys
sample_keys = [pool[i] for i in rng.choice(len(pool), size=min(SAMPLE_SIZE, len(pool)), replace=False)]
sample_mask = raw_train.set_index(['Store', 'Dept']).index.isin(sample_keys)
sample_df = raw_train[sample_mask].reset_index(drop=True)

tr_idx, va_idx = time_holdout(sample_df, weeks=8)
tr, va = sample_df.iloc[tr_idx], sample_df.iloc[va_idx]

grid_scores = {}
with mlflow.start_run(run_name=f'{MODEL_NAME}_CV'):
    mlflow.log_param('sample_size_series', len(sample_keys))
    for i, cfg in enumerate(candidate_configs):
        model = PerSeriesForecaster(**cfg)
        model.fit(tr, tr['Weekly_Sales'])
        pred = model.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        grid_scores[i] = score
        mlflow.log_metric(f'wmae_config_{i}', score)
        print(f'config {i} {cfg}: holdout WMAE={score:,.1f} (n_failed={model.n_failed_}/{model.n_series_})')
    best_i = min(grid_scores, key=grid_scores.get)
    best_config = candidate_configs[best_i]
    mlflow.log_param('best_config', best_config)
    mlflow.log_metric('best_wmae_sample', grid_scores[best_i])
print('best config:', best_config, '| WMAE:', grid_scores[best_i])

## Run 4 — Final fit + log Pipeline
Refit the chosen config on **all** series and log the Pipeline. Seasonal fits
across ~3,000 series will take noticeably longer than the ARIMA notebook —
expected, and exactly why the search above stayed small.

In [ ]:
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline

def make_pipeline():
    return Pipeline([('model', PerSeriesForecaster(**best_config))])

with mlflow.start_run(run_name=f'{MODEL_NAME}_Final') as run:
    mlflow.log_param('config', best_config)
    holdout_tr, holdout_va = time_holdout(raw_train, weeks=8)
    p = make_pipeline()
    p.fit(raw_train.iloc[holdout_tr], raw_train.iloc[holdout_tr]['Weekly_Sales'])
    hv = raw_train.iloc[holdout_va]
    holdout_pred = p.predict(hv)
    holdout_wmae = wmae(hv['Weekly_Sales'], holdout_pred, hv['IsHoliday'])
    mlflow.log_metric('holdout_wmae', holdout_wmae)

    final_pipe = make_pipeline()
    final_pipe.fit(raw_train, raw_train['Weekly_Sales'])
    example = raw_test.head(5)
    sig = infer_signature(example, final_pipe.predict(example))
    mlflow.sklearn.log_model(final_pipe, artifact_path='pipeline', signature=sig, input_example=example)
    print('holdout WMAE:', holdout_wmae, '| run_id:', run.info.run_id)

> **Registering the best model:** once you know this architecture is the overall best, register it from the Final run — either via the MLflow UI or:
> ```python
> mlflow.register_model(f'runs:/{run.info.run_id}/pipeline', 'walmart_best_model')
> ```

> **Comparing to ARIMA:** if SARIMA's CV WMAE isn't meaningfully better than plain
> ARIMA's, that itself is a useful theoretical finding worth writing up in the
> README — it would suggest the per-series seasonal signal is too short/noisy for
> SARIMA to exploit reliably at this history length, which the global tree/DL
> models (that borrow strength across all ~3,000 series at once) don't suffer from.